# A basic BBC News webscraper

In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.wait import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver import ActionChains
from bs4 import BeautifulSoup

In [2]:
# URL = 'https://www.bbc.co.uk/sport/football/68104699'
# URL = "https://www.bbc.co.uk/news/science-environment-41207827"
# URL = 'https://www.bbc.co.uk/sport/cricket/68128995'
URL = 'https://www.bbc.co.uk/sport/formula1/68163799'
WAIT_TIME = 20
COMMENTS_BUTTON_CLASS = 'view-comments-button'
MORE_COMMENTS_BUTTON_CLASS = 'comments__more-comments-button'
MORE_REPLIES_BUTTON_CLASS = 'reply-more-replies__button'
CONTENTS_CLASS = 'comment__content'
COMMENTS_TOTAL_CLASS = 'comments-total'
COMMENTS_TEXT_CLASS = {'class': 'comment__text'}
POST_DATE_CLASS = {'class': 'posted-date'}
POST_DATE_TAG = 'data-time'
UP_RATING_CLASS = {'class':'comment-rating__button-number'}
DOWN_RATING_CLASS = {'class':'comment__down-ratings'}

In [3]:
options = webdriver.ChromeOptions()
# options.add_argument('--headless')  # suppresses graphic rendering

In [4]:
driver = webdriver.Chrome(service=Service(), options=options)
driver.get(URL)

In [ ]:
wait = WebDriverWait(driver, WAIT_TIME)

## Click on 'View Comments' button

In [5]:
while True:
    try:
        comments_button = wait.until(EC.presence_of_element_located((By.CLASS_NAME, COMMENTS_BUTTON_CLASS)))
        ActionChains(driver).click(comments_button).perform()
    except:
        break
# This exposes all the "More comments" button

Break


## Repeatedly click on the 'More comments' button until it disappears

In [6]:
while True:
    try:
        # wait = WebDriverWait(driver, WAIT_TIME)
        more_comments_button = wait.until(EC.presence_of_element_located((By.CLASS_NAME, MORE_COMMENTS_BUTTON_CLASS)))
        ActionChains(driver).click(more_comments_button).perform()
    except:
        break
# This should expose all the "More replies" buttons

## Repeatedly click on the 'More replies' buttons

In [7]:
# this seems to work if the comments are closed
# but may cause issues if they're still open
more_replies_buttons = driver.find_elements(By.CLASS_NAME, MORE_REPLIES_BUTTON_CLASS)

for button in more_replies_buttons:
    while True:
        try:
            ActionChains(driver).click(button).perform()
        except:
            break

## Parse the html that's been exposed

In [8]:
bs = BeautifulSoup(driver.page_source, 'html.parser')
contents = bs.find_all('div', class_=CONTENTS_CLASS)
num_of_comments = int(bs.find('div', class_ = COMMENTS_TOTAL_CLASS).text.rstrip(' comments'))
# check that len(contents) == num_of_comments

## Tidy up

In [10]:
# tidy up by closing the driver
driver.close()

## Extract the data we're interested in

In [9]:
texts=[]
positives=[]
negatives=[]
dates=[]

for content in contents:
    texts.append(content.findChild('div', COMMENTS_TEXT_CLASS).text)
    dates.append(content.findChild('div', POST_DATE_CLASS)[POST_DATE_TAG])
    positives.append(content.findChild('div', UP_RATING_CLASS).text)
    negatives.append(content.findChild('div', DOWN_RATING_CLASS).text)